In [ ]:
%load_ext nbdistributed
%run _mgmn_nb_setup.py

# Initializing nvmath.distributed with torch.distributed

## 1) Initialize distributed workers with torch.distributed

In [ ]:
assert not using_mpi

# Initialize worker processes
%dist_init --num-processes 2 --gpu-ids 0,1

# disable distributed mode on cells without magic
%dist_mode --disable

using_torch_distributed = True

In [ ]:
%dist_status

In [ ]:
%%mgmn

print("hello world")

## 2) Initialize nvmath.distributed runtime

This code runs on every process.

In [ ]:
%%mgmn

import torch

import nvmath.distributed
from nvmath.distributed.process_group import TorchProcessGroup

device_id = rank % torch.cuda.device_count()
process_group = TorchProcessGroup(device_id=device_id)
nvmath.distributed.initialize(device_id, process_group, backends=["nccl"])
print("nvmath.distributed initialized")

## Finalize distributed runtime

In [ ]:
%%mgmn

import nvmath.distributed
nvmath.distributed.finalize()

In [ ]:
%dist_shutdown

using_torch_distributed = False

# Initializing nvmath.distributed with MPI

## 1) Initialize distributed workers with ipyparallel and MPI

In [ ]:
import ipyparallel as ipp

assert not using_torch_distributed

cluster = ipp.Cluster(engines="mpi", n=2)

using_mpi = True

# Start the cluster, block until ready, and connect to it
rc = cluster.start_and_connect_sync()
rc.activate()

## 2) Initialize nvmath.distributed runtime

This code runs on every process.

In [ ]:
%%mgmn

from cuda.core import system
from mpi4py import MPI

import nvmath.distributed
from nvmath.distributed.process_group import MPIProcessGroup

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
device_id = rank % system.get_num_devices()
process_group = MPIProcessGroup(comm)
nvmath.distributed.initialize(device_id, process_group, backends=["nccl"])
print("nvmath.distributed initialized")

## Finalize distributed runtime

In [ ]:
%%mgmn

import nvmath.distributed
nvmath.distributed.finalize()

In [ ]:
rc.shutdown(hub=True)

using_mpi = False

# Distributed matrix multiplication

In [ ]:
%%mgmn

import torch
import numpy as np
from nvmath.distributed.distribution import Slab
from nvmath.distributed.linalg.advanced import matrix_qualifiers_dtype

ctx = nvmath.distributed.get_context()
process_group = ctx.process_group

# The global problem size m, n, k
m, n, k = 256, 512, 256

row_wise_distribution = Slab.X
col_wise_distribution = Slab.Y

# Get my local shape
a_shape = col_wise_distribution.shape(process_group.rank, (m, k))
b_shape = col_wise_distribution.shape(process_group.rank, (n, k))
a = torch.rand(*a_shape, device=f"cuda:{ctx.device_id}")
b = torch.rand(*b_shape, device=f"cuda:{ctx.device_id}")

# Get a transposed view to obtain column-major memory layout. Note that this
# also changes the distribution of a and b.
a = a.T  # a is now (k, m) with row_wise_distribution
b = b.T  # b is now (k, n) with row_wise_distribution

# Distribution of a, b and output.
distributions = [row_wise_distribution, row_wise_distribution, col_wise_distribution]

qualifiers = np.zeros((3,), dtype=matrix_qualifiers_dtype)
qualifiers[0]["is_transpose"] = True  # a is transposed

# Use the stateful object as a context manager to automatically release resources.
result = nvmath.distributed.linalg.advanced.matmul(a, b, distributions=distributions, qualifiers=qualifiers)


In [ ]:
%%mgmn

result, result.shape